# Kaggle GPU Runner — Faster R-CNN 2-Session Parallel (F0 → F5)
Mở **2 tab Kaggle GPU T4 song song** (bật **Internet** + gắn Secret `WANDB_API_KEY`):
- **Tab 1**: đặt `SESSION = 1` (chạy EDA + Data Split + `F0`, `F1`, `F2` + Evaluate → tải `report_pack_session1.zip`)
- **Tab 2**: đặt `SESSION = 2` (chạy Data Split + `F3`, `F4 SAHI`, `F5 Tiled+SAHI`, `Grad-CAM`, `Video Demo` → tải `report_pack_session2.zip`)
- Hoặc đặt `SESSION = "all"` nếu muốn chạy tuần tự 1 phiên.

In [ ]:
SESSION = 1  # Đổi thành 2 ở Tab Kaggle thứ hai (hoặc "all")
BRANCH = "minh"
REPO_URL = "https://github.com/dohoangminh-vn/AIT3001.Group6.traffic.sign.detection.git"

%cd /kaggle/working
!rm -rf btl && git clone -b {BRANCH} {REPO_URL} btl
%cd /kaggle/working/btl
!pip install -q pycocotools sahi albumentations pyyaml
!python -c "import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

In [ ]:
import os
from pathlib import Path

# Tự động dò thư mục dataset Zalo AI trên /kaggle/input/
matches = list(Path("/kaggle/input").rglob("train_traffic_sign_dataset.json"))
assert matches, "Không tìm thấy train_traffic_sign_dataset.json trong /kaggle/input! Hãy Add Input dataset trước."
RAW_DIR = str(matches[0].parent)
IMG_DIR = f"{RAW_DIR}/images"
n_imgs = len(list(Path(IMG_DIR).glob("*.png")))
print(f"Detected RAW_DIR = {RAW_DIR} ({n_imgs} images)")
assert n_imgs == 4500, f"Expected 4500 images, got {n_imgs}"

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B API key loaded -> logging ONLINE (project: Traffic Sign).")
except Exception as e:
    print("W&B secret chưa gắn -> fallback OFFLINE + TensorBoard:", e)

In [ ]:
# Chuẩn hóa dữ liệu dùng chung (SHA256 cố định trên cả 2 tab) & chạy Unit Tests
if SESSION in (1, "all"):
    !python scripts/01_eda.py data.raw_dir={RAW_DIR}
!python scripts/02_prepare_data.py data.raw_dir={RAW_DIR}
!pytest -q

In [ ]:
# SESSION 1: F0 (Default), F1 (Small Sign AR), F2 (Warmup + ReduceLROnPlateau)
if SESSION in (1, "all"):
    !python scripts/04_train_frcnn.py img_dir={IMG_DIR} anchor_mode=default roi_size=7 loss_type=ce scheduler=cosine out_dir=runs/frcnn/f0_default
    !python scripts/04_train_frcnn.py img_dir={IMG_DIR} anchor_mode=small_sign_ar roi_size=7 loss_type=ce scheduler=cosine out_dir=runs/frcnn/f1_small_ar
    !python scripts/04_train_frcnn.py img_dir={IMG_DIR} anchor_mode=small_sign_ar roi_size=7 loss_type=ce scheduler=plateau out_dir=runs/frcnn/f2_plateau
    !python scripts/05_evaluate.py --img-dir {IMG_DIR} --check-oracle --preds \
      F0_Default=outputs/preds/f0_default_val.json \
      F1_SmallAR=outputs/preds/f1_small_ar_val.json \
      F2_Plateau=outputs/preds/f2_plateau_val.json
    !zip -rq /kaggle/working/report_pack_session1.zip outputs/eda outputs/eval outputs/preds runs/frcnn/f0_default/metrics_summary.json runs/frcnn/f1_small_ar/metrics_summary.json runs/frcnn/f2_plateau/metrics_summary.json runs/frcnn/*/resolved_config.yaml runs/frcnn/*/tb_logs
    !ls -lh /kaggle/working/report_pack_session1.zip

In [ ]:
# SESSION 2: F3 (CB-Focal + GIoU + RoI 14x14), F4 (Batch-4 SAHI), F5 (Tiled 640 + Copy-Paste + SAHI), Grad-CAM & Video
if SESSION in (2, "all"):
    !python scripts/04_train_frcnn.py img_dir={IMG_DIR} out_dir=runs/frcnn/f3_focal_roi14
    !python scripts/06_infer_sahi_demo.py img_dir={IMG_DIR} weights=runs/frcnn/f3_focal_roi14/best.pth
    !python scripts/04_train_frcnn.py img_dir={IMG_DIR} tiled=true copy_paste=true min_size=800 max_size=2048 out_dir=runs/frcnn/f5_tiled640
    !python scripts/06_infer_sahi_demo.py img_dir={IMG_DIR} weights=runs/frcnn/f5_tiled640/best.pth
    !python scripts/05_evaluate.py --img-dir {IMG_DIR} --check-oracle --preds \
      F3_CBFocal_RoI14=outputs/preds/f3_focal_roi14_val.json \
      F4_SAHI=outputs/preds/f3_focal_roi14_sahi_val.json \
      F5_Tiled_SAHI=outputs/preds/f5_tiled640_sahi_val.json
    !python scripts/07_analyze_and_demo.py --mode features img_dir={IMG_DIR} weights=runs/frcnn/f3_focal_roi14/best.pth
    !python scripts/07_analyze_and_demo.py --mode video img_dir={IMG_DIR} weights=runs/frcnn/f3_focal_roi14/best.pth
    !zip -rq /kaggle/working/report_pack_session2.zip outputs/eval outputs/preds runs/frcnn/f3_focal_roi14/metrics_summary.json runs/frcnn/f5_tiled640/metrics_summary.json runs/frcnn/*/resolved_config.yaml runs/frcnn/*/tb_logs
    !ls -lh /kaggle/working/report_pack_session2.zip